# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Metadata is an object, not a dict; use .to_json() for inspection
metadata = dataset.metadata.to_json()
print(f"Dataset Title: {metadata['name']}")
print(f"Description: {metadata['description']}\n")
print(f"Identifier: {metadata['identifier']}")
print(f"License: {metadata['license']}")
print(f"Published: {metadata['datePublished']}\n")
print(f"Available Keywords: {metadata['keywords']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, each record set and field is identified by its unique `@id`. Below we enumerate the record sets and their corresponding fields.

In [ ]:
# List all RecordSets with their @id and the fields within
record_sets = dataset.metadata.record_sets
print(f"Found {len(record_sets)} record sets:\n")
for i, rs in enumerate(record_sets):
    print(f"RecordSet {i+1}: @id = {rs.id}")
    print(f"  Name:    {rs.name}")
    print(f"  Fields and their @ids:")
    for field in rs.fields:
        print(f"    - {field.name} (@id={field.id}, Type={field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

> **Tip**: Replace the example RecordSet and Field `@id`s below with the ones printed above for your own advanced exploration.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    try:
        print(f"Loading records for RecordSet '@id': {record_set_id}")
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  --> {df.shape[0]} records, columns: {df.columns.tolist()}\n")
    except Exception as e:
        print(f"  Failed to load records for {record_set_id}: {e}\n")

# Pick the first available record set (as example for EDA and visualization)
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

if main_record_set_id:
    print(f"Columns of main record set ({main_record_set_id}):\n{dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Attempt to pick a numeric field from the columns (e.g., 'MW' or 'Coverage' or similar)
    # If you know a specific field @id from Section 2, put it here!
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Numeric field chosen for analysis: {numeric_field_id}")

        # Filter for values above average (or a fixed threshold if known)
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Optionally group by a categorical field
        group_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if group_candidates:
            group_field = group_candidates[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No main record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if main_record_set_id and 'numeric_field_id' in locals():
    plt.figure(figsize=(8,5))
    # Histogram for numeric field
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    # If grouping was done, show barplot
    if 'grouped_df' in locals() and grouped_df.shape[1] == 2:
        plt.figure(figsize=(12,5))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field_id])
        plt.title(f"Mean {numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.
This notebook demonstrated loading, overview, and initial exploratory analysis of the FAIR² mass spectrometry dataset using the [Croissant](https://mlcommons.org/croissant/) and [`mlcroissant`](https://github.com/mlcommons/croissant) Python library.

- The dataset provides detailed records of proteins in extracellular vesicles from human mast cells, including abundance, modifications, and sequence meta-data.
- Record sets and fields are referenced robustly via their Croissant `@id`s.
- Initial EDA steps help filter, normalize, and group numeric fields to prepare for further bioinformatics or ML analysis.

For advanced analysis, explore all available record sets and their fields as discovered in Section 2, and use field `@id`s for precise feature selection!